# Image format comparison for the low-bandwidth web: JPEG, WebP, AVIF and JPEG XL

Companion notebook for the paper *Choosing an Image Format for the Low-Bandwidth Web in 2026: An Experimental Comparison of JPEG, WebP, AVIF and JPEG XL on Photographs and Arabic-Text Images*. It runs the whole experiment on Google Colab:

1. Downloads the **Kodak** set (24 standard photographs) and generates **12 synthetic Arabic-text images** (optional).
2. Encodes every image in the four formats at several quality settings (AVIF in two chroma modes: 4:2:0 and 4:4:4).
3. Measures file size (bpp), quality (PSNR-Y, SSIM-Y and SSIMULACRA2), and encoding and decoding time.
4. Computes the **BD-rate** relative to JPEG, and the file size and download time of an image on slow connections.
5. Saves all tables and figures in the `results` folder and packs them into one zip file for download.

**How to run:** Runtime → Run all. Expected run time: about 30 minutes on the free tier of Colab.
If Colab asks to restart the session after the installation step, restart it and choose Run all again.

In [ ]:
# 1) Install the tools (fixed versions = reproducible results)
import subprocess, sys, os, platform

def pip_install(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

try:
    pip_install('imagecodecs==2026.10.10')   # libjpeg-turbo 3.2.0, libwebp 1.6.0, libavif 1.4.2 (aom), libjxl 0.12.0
except subprocess.CalledProcessError:
    print('WARNING: pinned imagecodecs not available for this Python; installing the latest version instead.')
    pip_install('imagecodecs')
# Pillow draws the Arabic-text images; its bundled FreeType 2.13.3, HarfBuzz 11.2.1 and libraqm 0.10.1
# make the images (and so every Arabic result) identical on any machine.
pip_install('pillow==11.3.0')
if 'PIL' in sys.modules and sys.modules['PIL'].__version__ != '11.3.0':
    print('Pillow was changed: use Runtime > Restart session, then Run all again.')
pip_install('scikit-image', 'scipy', 'pandas', 'matplotlib')

# Official SSIMULACRA2 tool from the JPEG XL project (libjxl v0.11.1 static Linux build)
JXL_TOOLS_URL = 'https://github.com/libjxl/libjxl/releases/download/v0.11.1/jxl-linux-x86_64-static-v0.11.1.tar.gz'
SSIMU2 = os.path.abspath('tools/ssimulacra2')
if not os.path.exists(SSIMU2):
    subprocess.run(['curl', '-sL', '-o', 'jxl_tools.tar.gz', JXL_TOOLS_URL], check=True)
    subprocess.run(['tar', 'xzf', 'jxl_tools.tar.gz', 'tools/ssimulacra2'], check=True)
    os.chmod(SSIMU2, 0o755)
print('SSIMULACRA2 tool ready:', os.path.exists(SSIMU2))

## 2) Settings
Change the values here only if needed. The defaults are the settings used in the paper.

In [ ]:
import imagecodecs, numpy as np

# Quality settings ("level") per format. Each list spans low to very high quality.
QUALITIES = {
    'JPEG':        [20, 30, 40, 50, 60, 70, 80, 90, 95],
    'WebP':        [20, 30, 40, 50, 60, 70, 80, 90, 95],
    'AVIF':        [20, 30, 40, 50, 60, 70, 80, 90],   # colour 4:2:0 (common default)
    'AVIF 4:4:4':  [20, 30, 40, 50, 60, 70, 80, 90],   # full colour resolution
    'JPEG XL':     [20, 30, 40, 50, 60, 70, 80, 90, 95],
}
REPEATS = 3                    # timing repeats per encode/decode (median is kept)
USE_ARABIC_TEXT = True         # also test 12 synthetic Arabic-text images
# SSIMULACRA2 targets for the size / download-time table (text images are already above ~70 at low settings)
TARGET_SCORES = {'Kodak': [60, 70, 80], 'ArabicText': [80]}
BANDWIDTHS_MBPS = [1, 2, 5, 10]
REF_WIDTH, REF_HEIGHT = 1920, 1080   # "typical web photo" used for the download-time table
OUT = 'results'
os.makedirs(OUT, exist_ok=True)

## 3) Download the Kodak images
The official source is r0k.us; if a download fails, a mirror on GitHub is used.

In [ ]:
import urllib.request
from PIL import Image

KODAK_DIR = 'data/kodak'
os.makedirs(KODAK_DIR, exist_ok=True)
SOURCES = [
    'http://r0k.us/graphics/kodak/kodak/kodim{n:02d}.png',
    'https://raw.githubusercontent.com/MohamedBakrAli/Kodak-Lossless-True-Color-Image-Suite/master/PhotoCD_PCD0992/{n:02d}.png',
]
failed_sources = set()          # a source that fails once is skipped for the remaining images
for n in range(1, 25):
    path = f'{KODAK_DIR}/kodim{n:02d}.png'
    if os.path.exists(path):
        continue
    for src in SOURCES:
        if src in failed_sources:
            continue
        try:
            with urllib.request.urlopen(src.format(n=n), timeout=30) as r, open(path, 'wb') as fh:
                fh.write(r.read())
            Image.open(path).verify()
            break
        except Exception:
            failed_sources.add(src)
            if os.path.exists(path):
                os.remove(path)
    if not os.path.exists(path):
        raise RuntimeError(f'Could not download kodim{n:02d}.png')
sizes = {Image.open(f'{KODAK_DIR}/kodim{n:02d}.png').size for n in range(1, 25)}
print('Kodak images:', len(os.listdir(KODAK_DIR)), 'sizes:', sizes)   # expect 24 and {(768, 512), (512, 768)}

## 4) Generate the Arabic-text images (optional)
Twelve 768×512 images modeled on university announcements and official pages: free Arabic fonts (SIL Open Font License),
different background colors and font sizes, and one image with full diacritics. The texts were written for this study.

In [ ]:
from PIL import ImageDraw, ImageFont, features

AR_DIR = 'data/arabic_text'
FONT_DIR = 'data/fonts'
FONT_URLS = {
    'Amiri': 'https://raw.githubusercontent.com/google/fonts/main/ofl/amiri/Amiri-Regular.ttf',
    'Amiri-Bold': 'https://raw.githubusercontent.com/google/fonts/main/ofl/amiri/Amiri-Bold.ttf',
    'NotoNaskh': 'https://raw.githubusercontent.com/google/fonts/main/ofl/notonaskharabic/NotoNaskhArabic%5Bwght%5D.ttf',
    'Cairo': 'https://raw.githubusercontent.com/google/fonts/main/ofl/cairo/Cairo%5Bslnt%2Cwght%5D.ttf',
}
TITLES = [
    'إعلان هام', 'جدول امتحانات منتصف الفصل', 'دعوة لحضور ورشة عمل', 'تعميم إداري',
    'نتائج القبول للعام الجامعي', 'أخبار الكلية', 'تنبيه للطلبة', 'مواعيد التسجيل',
    'محاضرة عامة', 'إعلان نتائج', 'تهنئة', 'حِكَمٌ عَرَبِيَّة',
]
BODY = [
    'تعلن إدارة الكلية عن بدء امتحانات منتصف الفصل يوم الأحد القادم، ويُرجى من جميع الطلبة الحضور قبل الموعد بنصف ساعة.',
    'تُعقد ورشة عمل بعنوان أساسيات ضغط الصور والفيديو في قاعة المحاضرات الرئيسية، والدعوة عامة لجميع المهتمين.',
    'يُرجى الالتزام بالتعليمات وإحضار البطاقة الجامعية، ولن يُسمح بالدخول بعد بدء الامتحان بربع ساعة.',
    'للاستفسار يمكن التواصل مع مكتب الدراسة والامتحانات خلال أوقات الدوام الرسمي من التاسعة صباحاً حتى الثانية ظهراً.',
    'الموعد: ١٥ / ١١ / ٢٠٢٦ — الساعة 10:30 صباحاً — المكان: المدرج رقم 3.',
]
DIACRITIC_BODY = [
    'مَنْ جَدَّ وَجَدَ، وَمَنْ زَرَعَ حَصَدَ، وَمَنْ سَارَ عَلَى الدَّرْبِ وَصَلَ.',
    'العِلْمُ نُورٌ، وَالصَّبْرُ مِفْتَاحُ الفَرَجِ، وَخَيْرُ الكَلَامِ مَا قَلَّ وَدَلَّ.',
    'القِرَاءَةُ غِذَاءُ العَقْلِ، وَالتَّعَلُّمُ رِحْلَةٌ لَا تَنْتَهِي.',
    'اِحْرِصْ عَلَى وَقْتِكَ، فَالوَقْتُ أَثْمَنُ مَا تَمْلِكُ.',
]
# (font, body size px, background, header colour, text colour, use diacritics)
LAYOUTS = [
    ('Amiri', 22, '#FFFFFF', '#1F5F3A', '#111111', False),
    ('NotoNaskh', 20, '#FFFFFF', '#7A1F2B', '#222222', False),
    ('Cairo', 22, '#F2F2F2', '#1F3A5F', '#111111', False),
    ('Amiri', 28, '#FFF8E7', '#8A5A00', '#1A1A1A', False),
    ('NotoNaskh', 18, '#FFFFFF', '#0B5394', '#333333', False),
    ('Cairo', 26, '#1F3A5F', '#F1C232', '#FFFFFF', False),
    ('Amiri', 20, '#FFFFFF', '#CC0000', '#CC0000', False),
    ('NotoNaskh', 24, '#EAF2F8', '#154360', '#0B0B0B', False),
    ('Cairo', 18, '#FFFFFF', '#38761D', '#202020', False),
    ('Amiri', 24, '#000000', '#FFD966', '#FFFFFF', False),
    ('NotoNaskh', 22, '#FDF2E9', '#A04000', '#111111', False),
    ('Amiri', 30, '#FFFFFF', '#5B2C6F', '#111111', True),
]

def get_fonts():
    os.makedirs(FONT_DIR, exist_ok=True)
    paths = {}
    for name, url in FONT_URLS.items():
        p = f'{FONT_DIR}/{name}.ttf'
        if not os.path.exists(p):
            with urllib.request.urlopen(url, timeout=60) as r, open(p, 'wb') as fh:
                fh.write(r.read())
        paths[name] = p
    return paths

USE_RAQM = features.check('raqm')   # proper Arabic shaping inside Pillow
if not USE_RAQM:
    pip_install('arabic-reshaper', 'python-bidi')
    import arabic_reshaper
    from bidi.algorithm import get_display

def load_font(path, size):
    if USE_RAQM:
        return ImageFont.truetype(path, size, layout_engine=ImageFont.Layout.RAQM)
    return ImageFont.truetype(path, size, layout_engine=ImageFont.Layout.BASIC)

def draw_rtl(draw, x_right, y, text, font, fill):
    if USE_RAQM:
        draw.text((x_right, y), text, font=font, fill=fill, anchor='ra', direction='rtl', language='ar')
    else:
        draw.text((x_right, y), get_display(arabic_reshaper.reshape(text)), font=font, fill=fill, anchor='ra')

def text_width(font, text):
    if USE_RAQM:
        return font.getlength(text, direction='rtl', language='ar')
    return font.getlength(get_display(arabic_reshaper.reshape(text)))

def wrap(font, text, max_w):
    lines, cur = [], ''
    for word in text.split():
        trial = (cur + ' ' + word).strip()
        if text_width(font, trial) <= max_w:
            cur = trial
        else:
            lines.append(cur)
            cur = word
    if cur:
        lines.append(cur)
    return lines

def make_arabic_images():
    os.makedirs(AR_DIR, exist_ok=True)
    fonts = get_fonts()
    W, H, M = 768, 512, 36
    for i, (fname, size, bg, header, fg, diac) in enumerate(LAYOUTS):
        img = Image.new('RGB', (W, H), bg)
        d = ImageDraw.Draw(img)
        d.rectangle([0, 0, W, 86], fill=header)                    # coloured header band
        d.ellipse([M - 6, 18, M + 44, 68], outline='#FFFFFF', width=3)  # simple round "logo"
        title_font = load_font(fonts['Amiri-Bold'], 40)
        title_fill = '#000000' if header in ('#F1C232', '#FFD966') else '#FFFFFF'
        draw_rtl(d, W - M, 20, TITLES[i], title_font, title_fill)
        body_font = load_font(fonts[fname], size)
        paragraphs = DIACRITIC_BODY if diac else [BODY[(i + k) % len(BODY)] for k in range(4)]
        y, lh = 110, int(size * 1.9)
        for para in paragraphs:
            lines = wrap(body_font, para, W - 2 * M)
            if y + len(lines) * lh > H - 44:      # draw whole paragraphs only
                break
            for line in lines:
                draw_rtl(d, W - M, y, line, body_font, fg)
                y += lh
            y += int(size * 0.6)
        d.line([M, H - 36, W - M, H - 36], fill=header, width=2)    # footer rule
        small = load_font(fonts['NotoNaskh'], 14)
        draw_rtl(d, W - M, H - 30, 'كلية تقنية المعلومات — قسم هندسة البرمجيات', small, fg if bg != '#FFFFFF' else '#555555')
        img.save(f'{AR_DIR}/arabic{i + 1:02d}.png')
    print('Arabic text images:', len(os.listdir(AR_DIR)), '| RAQM shaping:', USE_RAQM)

if USE_ARABIC_TEXT:
    make_arabic_images()

def contact_sheet(files, path, cols=4, thumb=(384, 256)):
    # a grid of thumbnails of the test images
    rows_n = (len(files) + cols - 1) // cols
    sheet = Image.new('RGB', (cols * thumb[0], rows_n * thumb[1]), 'white')
    for k, f in enumerate(files):
        im = Image.open(f).convert('RGB')
        im.thumbnail(thumb)
        r, c = divmod(k, cols)
        sheet.paste(im, (c * thumb[0] + (thumb[0] - im.width) // 2, r * thumb[1] + (thumb[1] - im.height) // 2))
    sheet.save(path)

import glob
contact_sheet(sorted(glob.glob(f'{KODAK_DIR}/*.png'))[:8], f'{OUT}/samples_kodak.png')
if USE_ARABIC_TEXT:
    contact_sheet(sorted(glob.glob(f'{AR_DIR}/*.png')), f'{OUT}/samples_arabic_text.png')
print('Saved sample sheets of the test images')

## 5) The four formats (encoders and settings)
Each encoder runs at the default speed or effort of its reference command-line tool. JPEG, WebP and AVIF use 4:2:0 chroma subsampling,
and every encoder and decoder runs on a single thread so that speeds are comparable.
AVIF is also encoded with 4:4:4 chroma (no subsampling) to measure the effect of subsampling, especially on colored text.

In [ ]:
AVIF420 = imagecodecs.AVIF.PIXEL_FORMAT.YUV420
AVIF444 = imagecodecs.AVIF.PIXEL_FORMAT.YUV444

CODECS = {
    'JPEG':    (lambda im, q: imagecodecs.jpeg8_encode(im, level=q, optimize=True),          # libjpeg-turbo, 4:2:0
                lambda b: imagecodecs.jpeg8_decode(b)),
    'WebP':    (lambda im, q: imagecodecs.webp_encode(im, level=q, lossless=False, method=4),  # lossless=False is essential
                lambda b: imagecodecs.webp_decode(b)),
    'AVIF':    (lambda im, q: imagecodecs.avif_encode(im, level=q, speed=6, pixelformat=AVIF420, numthreads=1),  # libavif + aom
                lambda b: imagecodecs.avif_decode(b, numthreads=1)),
    'AVIF 4:4:4': (lambda im, q: imagecodecs.avif_encode(im, level=q, speed=6, pixelformat=AVIF444, numthreads=1),
                lambda b: imagecodecs.avif_decode(b, numthreads=1)),
    'JPEG XL': (lambda im, q: imagecodecs.jpegxl_encode(im, level=q, effort=7, numthreads=1),  # libjxl, default effort 7
                lambda b: imagecodecs.jpegxl_decode(b, numthreads=1)),
}
EXT = {'JPEG': 'jpg', 'WebP': 'webp', 'AVIF': 'avif', 'AVIF 4:4:4': 'avif', 'JPEG XL': 'jxl'}

## 6) Quality metrics
- **PSNR-Y** and **SSIM-Y**: computed on the luma channel Y, the usual practice in codec comparisons.
- **SSIMULACRA2**: a recent perceptual metric (100 = identical to the original); the official tool from the JPEG XL project is used.
- For the curves and the BD-rate, SSIM-Y is converted to decibels, −10·log10(1 − SSIM), because SSIM saturates near 1 (as recommended by Herglotz et al., IEEE TIP 2024).

In [ ]:
import time, tempfile
from skimage.metrics import structural_similarity

def to_y(rgb):
    rgb = rgb.astype(np.float64)
    return 0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2]   # BT.601 luma

def psnr_y(a, b):
    mse = np.mean((to_y(a) - to_y(b)) ** 2)
    return float('inf') if mse == 0 else 10 * np.log10(255.0 ** 2 / mse)

def ssim_y(a, b):
    return structural_similarity(to_y(a), to_y(b), data_range=255, gaussian_weights=True,
                                 sigma=1.5, use_sample_covariance=False)

def ssimulacra2(ref_png, dist_rgb, tmpdir):
    p = os.path.join(tmpdir, 'dist.png')
    Image.fromarray(dist_rgb).save(p, compress_level=1)
    out = subprocess.run([SSIMU2, ref_png, p], capture_output=True, text=True, check=True)
    return float(out.stdout.strip())

def timed(fn, *args):
    times, out = [], None
    for _ in range(REPEATS):
        t0 = time.perf_counter()
        out = fn(*args)
        times.append(time.perf_counter() - t0)
    return out, float(np.median(times))

## 7) Run the experiment
Results are appended to `results/raw_results.csv` after each image; if the run is interrupted, run this cell again and it continues where it stopped.

In [ ]:
import pandas as pd, glob

datasets = {'Kodak': sorted(glob.glob(f'{KODAK_DIR}/*.png'))}
if USE_ARABIC_TEXT:
    datasets['ArabicText'] = sorted(glob.glob(f'{AR_DIR}/*.png'))

RAW = f'{OUT}/raw_results.csv'
done = set()
if os.path.exists(RAW):
    prev = pd.read_csv(RAW)
    done = set(zip(prev.dataset, prev.image, prev.codec, prev.quality))
rows = []
tmpdir = tempfile.mkdtemp()
total = sum(len(v) for v in datasets.values()) * sum(len(q) for q in QUALITIES.values())
count, t_start = 0, time.time()
for ds, files in datasets.items():
    for f in files:
        name = os.path.basename(f)
        img = np.asarray(Image.open(f).convert('RGB'))
        h, w = img.shape[:2]
        for codec, (enc, dec) in CODECS.items():
            for q in QUALITIES[codec]:
                count += 1
                if (ds, name, codec, q) in done:
                    continue
                data, t_enc = timed(enc, img, q)
                out, t_dec = timed(dec, data)
                out = np.ascontiguousarray(np.asarray(out)[..., :3], dtype=np.uint8)
                rows.append(dict(dataset=ds, image=name, width=w, height=h, codec=codec, quality=q,
                                 bytes=len(data), bpp=8 * len(data) / (w * h),
                                 psnr_y=psnr_y(img, out), ssim_y=ssim_y(img, out),
                                 ssimulacra2=ssimulacra2(f, out, tmpdir),
                                 enc_ms=1000 * t_enc, dec_ms=1000 * t_dec))
        pd.DataFrame(rows).to_csv(RAW, mode='a', header=not os.path.exists(RAW), index=False)
        rows = []
        print(f'{ds}/{name} done  ({count}/{total} encodes, {(time.time() - t_start) / 60:.1f} min)')
df = pd.read_csv(RAW)
df['ssim_y_db'] = -10 * np.log10(np.clip(1 - df.ssim_y, 1e-10, None))   # SSIM-Y in dB (used for curves and BD-rate)
print('Rows:', len(df))

## 8) Mean values per format and quality setting

In [ ]:
ORDER = list(CODECS)
summary = (df.groupby(['dataset', 'codec', 'quality'])
             .agg(bpp=('bpp', 'mean'), psnr_y=('psnr_y', 'mean'), ssim_y=('ssim_y', 'mean'),
                  ssim_y_db=('ssim_y_db', 'mean'), ssimulacra2=('ssimulacra2', 'mean'),
                  enc_ms=('enc_ms', 'median'), dec_ms=('dec_ms', 'median'))
             .reset_index())
summary['codec'] = pd.Categorical(summary.codec, ORDER, ordered=True)
summary = summary.sort_values(['dataset', 'codec', 'quality'])
summary.to_csv(f'{OUT}/summary_by_quality.csv', index=False)
summary.round(3)

## 9) Rate–quality (RD) curves
Each point is the mean over all images at one quality setting. Higher and further to the left is better.

In [ ]:
import matplotlib.pyplot as plt

COLORS = {'JPEG': '#7f7f7f', 'WebP': '#2ca02c', 'AVIF': '#d62728', 'AVIF 4:4:4': '#ff9896', 'JPEG XL': '#1f77b4'}
MARKERS = {'JPEG': 'o', 'WebP': 's', 'AVIF': '^', 'AVIF 4:4:4': 'v', 'JPEG XL': 'D'}
METRICS = [('psnr_y', 'PSNR-Y (dB)'), ('ssim_y_db', 'SSIM-Y (dB)'), ('ssimulacra2', 'SSIMULACRA2')]
for ds in summary.dataset.unique():
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
    for ax, (m, label) in zip(axes, METRICS):
        for codec in ORDER:
            s = summary[(summary.dataset == ds) & (summary.codec == codec)]
            ax.plot(s.bpp, s[m], marker=MARKERS[codec], color=COLORS[codec], label=codec, lw=1.6, ms=5)
        ax.set_xlabel('bits per pixel (bpp)')
        ax.set_ylabel(label)
        ax.grid(alpha=0.3)
        ax.set_xlim(0, min(4, summary[summary.dataset == ds].bpp.max() * 1.02))
    axes[0].legend()
    fig.suptitle(f'Rate-quality curves - {ds} ({df[df.dataset == ds].image.nunique()} images)')
    fig.tight_layout()
    fig.savefig(f'{OUT}/rd_curves_{ds}.png', dpi=300)
    plt.show()

## 10) BD-rate relative to JPEG
A negative value means that the format needs fewer bits than JPEG for the same quality; for example, −40% means a 40% smaller file.
The BD-rate is computed for each image separately (Bjøntegaard method with PCHIP interpolation), then the mean and median are reported. SSIM-Y is used in decibels.

In [ ]:
from scipy.interpolate import PchipInterpolator

def bd_rate(rate_a, qual_a, rate_b, qual_b):
    """Average bitrate difference (%) of B relative to A over their common quality range."""
    qa, ra = np.asarray(qual_a, float), np.log(np.asarray(rate_a, float))
    qb, rb = np.asarray(qual_b, float), np.log(np.asarray(rate_b, float))
    ia, ib = np.argsort(qa), np.argsort(qb)
    qa, ra, qb, rb = qa[ia], ra[ia], qb[ib], rb[ib]
    if np.any(np.diff(qa) <= 0) or np.any(np.diff(qb) <= 0):
        return np.nan                      # quality must increase with rate
    lo, hi = max(qa[0], qb[0]), min(qa[-1], qb[-1])
    if hi <= lo:
        return np.nan                      # no common quality range
    ia_ = PchipInterpolator(qa, ra).integrate(lo, hi)
    ib_ = PchipInterpolator(qb, rb).integrate(lo, hi)
    return (np.exp((ib_ - ia_) / (hi - lo)) - 1) * 100

bd_rows = []
for (ds, image), g in df.groupby(['dataset', 'image']):
    ref = g[g.codec == 'JPEG'].sort_values('quality')
    for codec in ORDER[1:]:
        t = g[g.codec == codec].sort_values('quality')
        for m, _ in METRICS:
            bd_rows.append(dict(dataset=ds, image=image, codec=codec, metric=m,
                                bd_rate=bd_rate(ref.bpp, ref[m], t.bpp, t[m])))
bd = pd.DataFrame(bd_rows)
bd.to_csv(f'{OUT}/bd_rate_per_image.csv', index=False)
missing = bd[bd.bd_rate.isna()]
if len(missing):
    print('No common quality range (excluded from the averages):')
    print(missing[['dataset', 'image', 'codec', 'metric']].to_string(index=False))
bd_table = (bd.groupby(['dataset', 'codec', 'metric']).bd_rate
              .agg(mean='mean', median='median', valid_images='count').reset_index())
bd_table.to_csv(f'{OUT}/bd_rate_vs_jpeg.csv', index=False)
bd_table.pivot_table(index=['dataset', 'codec'], columns='metric', values='mean').round(1)

## 11) Size at a target quality and download time on a slow connection
For each format, the bpp needed to reach an SSIMULACRA2 score of 60, 70 or 80 (Kodak) or 80 (Arabic text) is interpolated,
then converted to the size of a 1920×1080 image and its download time.

In [ ]:
def bpp_at_score(g, target):
    g = g.sort_values('ssimulacra2')
    s, r = g.ssimulacra2.values, np.log(g.bpp.values)
    if np.any(np.diff(s) <= 0) or not (s[0] <= target <= s[-1]):
        return np.nan
    return float(np.exp(PchipInterpolator(s, r)(target)))

tq_rows = []
for (ds, image, codec), g in df.groupby(['dataset', 'image', 'codec']):
    for t in TARGET_SCORES.get(ds, []):
        tq_rows.append(dict(dataset=ds, image=image, codec=codec, target=t, bpp=bpp_at_score(g, t)))
tq = pd.DataFrame(tq_rows)
# keep only images where EVERY format reaches the target, so all formats are compared on the same images
ok = tq.groupby(['dataset', 'target', 'image']).bpp.apply(lambda x: x.notna().all()).rename('paired').reset_index()
tq = tq.merge(ok, on=['dataset', 'target', 'image'])
tq = tq[tq.paired]
target_table = (tq.groupby(['dataset', 'target', 'codec']).bpp
                  .agg(bpp='mean', images='count').reset_index())
pixels = REF_WIDTH * REF_HEIGHT
target_table['kB_for_1920x1080'] = target_table.bpp * pixels / 8 / 1000
for bw in BANDWIDTHS_MBPS:
    target_table[f'seconds_at_{bw}Mbps'] = target_table.bpp * pixels / (bw * 1e6)
jpeg_ref = target_table[target_table.codec == 'JPEG'].set_index(['dataset', 'target']).bpp
target_table['saving_vs_JPEG_%'] = [100 * (1 - r.bpp / jpeg_ref.loc[(r.dataset, r.target)])
                                    for r in target_table.itertuples()]
target_table['codec'] = pd.Categorical(target_table.codec, ORDER, ordered=True)
target_table = target_table.sort_values(['dataset', 'target', 'codec'])
target_table.to_csv(f'{OUT}/size_and_download_time.csv', index=False)
target_table.round(2)

## 12) Speed: encoding and decoding time
Median per format over all images and quality settings, on a single thread. Megapixels per second = millions of pixels processed per second.

In [ ]:
df['megapixels'] = df.width * df.height / 1e6
speed = (df.groupby(['dataset', 'codec'])
           .agg(enc_ms_median=('enc_ms', 'median'), dec_ms_median=('dec_ms', 'median'),
                megapixels=('megapixels', 'mean')).reset_index())
speed['enc_MP_per_s'] = speed.megapixels / (speed.enc_ms_median / 1000)
speed['dec_MP_per_s'] = speed.megapixels / (speed.dec_ms_median / 1000)
speed['codec'] = pd.Categorical(speed.codec, ORDER, ordered=True)
speed = speed.sort_values(['dataset', 'codec'])
speed.to_csv(f'{OUT}/speed.csv', index=False)
speed.round(2)

## 13) Visual comparison at about the same size
A crop of kodim23 for each format, from the file closest to 0.5 bpp (a qualitative figure for the paper).

In [ ]:
def visual_comparison(image_name='kodim23.png', target_bpp=0.5, box=(120, 170, 290, 300)):
    src = f'{KODAK_DIR}/{image_name}'
    img = np.asarray(Image.open(src).convert('RGB'))
    fig, axes = plt.subplots(1, len(ORDER) + 1, figsize=(3.2 * (len(ORDER) + 1), 3.6))
    axes[0].imshow(img[box[1]:box[3], box[0]:box[2]])
    axes[0].set_title('Original')
    for ax, codec in zip(axes[1:], ORDER):
        g = df[(df.dataset == 'Kodak') & (df.image == image_name) & (df.codec == codec)]
        q = int(g.iloc[(g.bpp - target_bpp).abs().argsort().iloc[0]].quality)
        enc, dec = CODECS[codec]
        data = enc(img, q)
        out = np.asarray(dec(data))[..., :3]
        r = g[g.quality == q].iloc[0]
        ax.imshow(out[box[1]:box[3], box[0]:box[2]])
        ax.set_title(f'{codec} q={q}\n{8 * len(data) / img[..., 0].size:.2f} bpp, SSIMU2={r.ssimulacra2:.1f}', fontsize=9)
    for ax in axes:
        ax.axis('off')
    fig.tight_layout()
    fig.savefig(f'{OUT}/visual_comparison.png', dpi=300)
    plt.show()

visual_comparison()

## 14) Environment information (versions reported in the paper)

In [ ]:
v = imagecodecs.version(dict)
env = {
    'python': platform.python_version(), 'machine': platform.machine(), 'processor': platform.processor(),
    'cpu_count': os.cpu_count(), 'imagecodecs': v.get('imagecodecs'), 'libjpeg_turbo': v.get('libjpeg_turbo'),
    'libwebp': v.get('libwebp'), 'libavif': v.get('libavif'), 'libjxl': v.get('libjxl'),
    'ssimulacra2_tool': 'libjxl v0.11.1 static release', 'repeats': REPEATS,
}
try:
    env['cpu_model'] = [l.split(':', 1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name')][0]
except Exception:
    pass
import PIL
from PIL import features
env.update({'pillow': PIL.__version__, 'freetype': features.version('freetype2'),
            'harfbuzz': features.version('harfbuzz'), 'raqm': features.version('raqm')})
try:   # AV1 codecs inside libavif (encoder and decoder actually used)
    import ctypes
    lib = ctypes.CDLL(glob.glob(os.path.join(os.path.dirname(imagecodecs.__file__), '..',
                                             'imagecodecs.libs', 'libavif-*.so*'))[0])
    buf = ctypes.create_string_buffer(256)
    lib.avifCodecVersions(buf)
    env['avif_codecs'] = buf.value.decode()
except Exception:
    pass
pd.Series(env).to_csv(f'{OUT}/environment.csv')
pd.Series(env)

## 15) Download all results
Creates `results.zip` (CSV tables and PNG figures). On Colab the download starts automatically.

In [ ]:
import shutil
shutil.make_archive('results', 'zip', OUT)
try:
    from google.colab import files
    files.download('results.zip')
except ImportError:
    print('Saved results.zip')